# FinGPT-Style Evaluation on Gold Commodity Dataset (Groq API Version)

This notebook mirrors the structure of your FinBERT notebook, but evaluates a **FinGPT-style sentiment workflow** on the Gold Commodity dataset using the **Groq API**.

It is set up so you can fill in your own key and model choice.


## 0. Installation

In [ ]:

%pip install -U openai datasets scikit-learn pandas torch

## Device Check

In [3]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")


=== nvidia-smi (if available) ===
nvidia-smi not found on this machine.
Detected runtime device: cpu


## 1. Environment and Reproducibility Setup

In [4]:
import random

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paste your Groq key directly below.
# Do not share this notebook or upload it publicly after adding your real key.
GROQ_API_KEY = "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"

print("GROQ_API_KEY set:", GROQ_API_KEY != "your_groq_api_key_here" and bool(GROQ_API_KEY))


GROQ_API_KEY set: True


## 2. Dataset Loading

In [2]:
from datasets import load_dataset
import pandas as pd

gold_ds = load_dataset("SaguaroCapital/sentiment-analysis-in-commodity-market-gold")

split_name = "test" if "test" in gold_ds else "train"
gold_raw_df = gold_ds[split_name].to_pandas()

# Explicitly set the correct columns for this dataset
sentence_col = "News"
label_col = "Price Sentiment"

gold_raw_df = gold_raw_df[[sentence_col, label_col]].rename(
    columns={sentence_col: "text", label_col: "true_label"}
).dropna()

# Optional: standardize labels
gold_raw_df["true_label"] = gold_raw_df["true_label"].astype(str).str.strip().str.lower()

gold_raw_df.head()

,text,true_label
0,Gold / Silver / Copper futures - weekly outloo...,none
1,gold to be a safe-haven again; sell crude on r...,none
2,"feb. gold settles at $1,097,90/oz on comex, do...",negative
3,dec gold rises 30c to $443.40/oz in morning ny...,positive
4,Gold holds modest losses after Chicago PMI miss,negative


## 3. FinGPT-Style Model Setup via Groq

This notebook uses a **prompted chat-completion approach** instead of a classifier head, so it behaves more like a FinGPT-style instruction workflow.

Swap `MODEL_NAME` to any Groq-hosted model you want to test.


In [4]:
import re
from openai import OpenAI

GROQ_API_KEY = "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"

if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError("Replace the placeholder in GROQ_API_KEY with your actual Groq API key before running.")

MODEL_NAME = "llama-3.1-8b-instant"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

print(f"Using Groq model: {MODEL_NAME}")

Using Groq model: llama-3.1-8b-instant


## 4. Prompt Template and Prediction Parser

In [5]:
LABELS = ["negative", "neutral", "positive"]

SYSTEM_PROMPT = (
    "You are a financial sentiment classifier. "
    "Classify the sentiment of the given financial or commodity-market text. "
    "Return only one lowercase label from: negative, neutral, positive."
)

def build_prompt(text: str) -> str:
    return f"""Determine the sentiment of this financial text.

Text: {text}

Return only one label from: negative, neutral, positive."""

def parse_label(output_text: str) -> str:
    cleaned = output_text.strip().lower()

    for label in ["negative", "neutral", "positive"]:
        if re.search(rf'\b{label}\b', cleaned):
            return label

    if "bearish" in cleaned or "loss" in cleaned or "drop" in cleaned:
        return "negative"
    if "bullish" in cleaned or "gain" in cleaned or "rise" in cleaned:
        return "positive"

    return "neutral"


## 5. Batched Inference

In [6]:
def groq_predict_one(text: str, temperature: float = 0) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(text)},
        ],
    )
    raw = response.choices[0].message.content.strip()
    return parse_label(raw)

def run_fingpt_batched(sentences, batch_size=8, temperature=0):
    # Groq chat completions are called one prompt at a time here.
    # batch_size is kept only to mirror the structure of your earlier notebook.
    preds = []
    total = len(sentences)

    for i, text in enumerate(sentences, start=1):
        pred = groq_predict_one(text, temperature=temperature)
        preds.append(pred)

        if i % batch_size == 0 or i == total:
            print(f"Processed {i}/{total}")

    return preds


## 6. Full Evaluation Metrics

In [8]:
from sklearn.metrics import classification_report, f1_score, accuracy_score, matthews_corrcoef

# keep only first 200 rows
gold_eval_df = gold_raw_df.head(200).copy()

true_labels = gold_eval_df["true_label"].tolist()
pred_labels = run_fingpt_batched(
    gold_eval_df["text"].tolist(),
    batch_size=20,
    temperature=0
)

metrics = {
    "macro_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="macro", zero_division=0),
    "micro_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="micro", zero_division=0),
    "weighted_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="weighted", zero_division=0),
    "accuracy": accuracy_score(true_labels, pred_labels),
    "mcc": matthews_corrcoef(true_labels, pred_labels),
}

metrics_df = pd.DataFrame([metrics])
print(metrics_df)

print("\nClassification Report:\n")
print(classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0))

Processed 20/200
Processed 40/200
Processed 60/200
Processed 80/200
Processed 100/200
Processed 120/200
Processed 140/200
Processed 160/200
Processed 180/200
Processed 200/200
   macro_f1  micro_f1  weighted_f1  accuracy       mcc
0  0.515547  0.553073     0.685103     0.495  0.418464

Classification Report:

              precision    recall  f1-score   support

    negative       0.81      0.70      0.75        69
     neutral       0.07      1.00      0.12         6
    positive       0.88      0.54      0.67        83

   micro avg       0.49      0.63      0.55       158
   macro avg       0.59      0.75      0.52       158
weighted avg       0.82      0.63      0.69       158



## 7. Save Predictions

In [10]:
from pathlib import Path

EXPORT_DIR = Path("experiments/sentiment_agent/fingpt_style_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pred_df = gold_eval_df.copy()   # use the 200-row subset
pred_df["pred_label"] = pred_labels
pred_df["model_name"] = MODEL_NAME
pred_df["provider"] = "groq"

csv_path = EXPORT_DIR / "gold_fingpt_style_predictions_groq.csv"
pred_df.to_csv(csv_path, index=False)

print(f"Saved predictions to: {csv_path}")

Saved predictions to: experiments\sentiment_agent\fingpt_style_outputs\gold_fingpt_style_predictions_groq.csv


## 8. Optional Save of Run Metadata

Since this notebook uses API inference, there is no local model/tokenizer to save.
Instead, this cell saves the run configuration for reproducibility.


In [12]:
from pathlib import Path
import json

MODEL_EXPORT_DIR = Path("experiments/sentiment_agent/saved_models/fingpt_style_groq_run")
MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

run_config = {
    "provider": "groq",
    "model_name": MODEL_NAME,
    "seed": 42,
    "dataset": "SaguaroCapital/sentiment-analysis-in-commodity-market-gold",
    "labels": LABELS,
}

config_path = MODEL_EXPORT_DIR / "run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print(f"Saved run metadata to: {config_path}")

Saved run metadata to: experiments\sentiment_agent\saved_models\fingpt_style_groq_run\run_config.json


## 9. Cleanup

In [14]:
import gc

for name in ["client"]:
    if name in globals():
        del globals()[name]

gc.collect()

if "torch" in globals():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Cleanup complete.")

Cleanup complete.
